In [ ]:
!pip install pyhealth

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
from pyhealth.datasets import MIMIC3Dataset

dataset = MIMIC3Dataset(
    root='/content/drive/MyDrive/mimic-iii-clinical-database-1.4 3',
    tables=["DIAGNOSES_ICD", "PROCEDURES_ICD", "PRESCRIPTIONS"],
    dev= True
)

dataset.stats()

In [ ]:
# Helper functions
from pyhealth.data import Patient, Event
from datetime import datetime


def getDOB(P: Patient):
  return P.get_events(event_type='patients')[0].attr_dict['dob']


def getDischTime(E: Event):
  return E.attr_dict['dischtime']

def convertToDT(dtString: str):
  format = '%Y-%m-%d %H:%M:%S'
  dt = datetime.strptime(dtString, format)
  return dt

def getAge(dob: datetime, admissionDate: datetime):
  age = (admissionDate - dob).days // 365
  return age

def getHADID(E: Event):
  return E.attr_dict['hadm_id']

def bucket_time_gap(days):
  if days <= 1:
    return 0
  elif days <= 7:
    return 1
  elif days <= 30:
    return 2
  elif days <= 60:
    return 3
  elif days <= 90:
    return 4
  else:
    return 5

def bucket_age(age):
  return age // 10


In [ ]:
from collections import defaultdict
from pyhealth.tasks import BaseTask
from pyhealth.data import Patient
from typing import List, Dict
from pyhealth.processors import BinaryLabelProcessor, SequenceProcessor, NestedSequenceProcessor, MultiLabelProcessor
import polars as pl

class BEHRTTask(BaseTask):
    def __init__(self, task_type="readmission"):
      self.task_type = task_type
      self.input_schema: Dict[str, str] = {
          "codes": NestedSequenceProcessor,
          "ages": SequenceProcessor,
          "time_gap": SequenceProcessor,
      }
      if task_type == "readmission":
          self.output_schema = {
              "readmission": BinaryLabelProcessor
          }
      elif task_type == "drug":
          self.input_schema["drugs_hist"] = NestedSequenceProcessor
          self.output_schema = {
              "drugs": MultiLabelProcessor
          }

    def pre_filter(self, df: pl.LazyFrame) -> pl.LazyFrame:

      has_diagnoses = (
          df.filter(pl.col("event_type") == "diagnoses_icd")
          .select("patient_id")
          .unique()
          .collect()
          .to_series()
      )

      has_dob = (
          df.filter(
              (pl.col("event_type") == "patients")
              & (pl.col("patients/dob").is_not_null())
          )
          .select("patient_id")
          .unique()
          .collect()
          .to_series()
      )

      died_in_hospital = (
          df.filter(
              (pl.col("event_type") == "admissions")
              & (pl.col("admissions/hospital_expire_flag") == "1")
          )
          .select("patient_id")
          .unique()
          .collect()
          .to_series()
      )

      has_prescriptions = (
          df.filter(pl.col("event_type") == "prescriptions")
          .select("patient_id")
          .unique()
          .collect()
          .to_series()
      )

      has_procedures = (
          df.filter(pl.col("event_type") == "procedures_icd")
          .select("patient_id")
          .unique()
          .collect()
          .to_series()
      )

      condition = (
          pl.col("patient_id").is_in(has_diagnoses)
          & pl.col("patient_id").is_in(has_dob)
      )

      if self.task_type == "drug":
        condition = (
          condition
          & pl.col("patient_id").is_in(has_procedures)
          & pl.col("patient_id").is_in(has_prescriptions)
        )
      elif self.task_type == "readmission":
        condition = (
          condition
          & ~pl.col("patient_id").is_in(died_in_hospital)
        )

      filtered_df = df.filter(condition)
      return filtered_df

    def __call__(self, patient: Patient) -> List[Dict]:
        samples = []

        dob_dt = convertToDT(getDOB(patient))

        medications = patient.get_events(event_type="prescriptions")
        diagnoses = patient.get_events(event_type='diagnoses_icd')
        procedures = patient.get_events(event_type='procedures_icd')
        admissions = patient.get_events(event_type='admissions')

        visit_codes = defaultdict(list)
        for diag in diagnoses:
            visit_codes[getHADID(diag)].append(diag.attr_dict['icd9_code'])
        for proc in procedures:
            visit_codes[getHADID(proc)].append(proc.attr_dict['icd9_code'])

        visit_medications = defaultdict(list)
        for med in medications:
            visit_medications[getHADID(med)].append(med.attr_dict['drug'])

        sorted_admissions = sorted(admissions, key=lambda e: e.timestamp)


        for i, admission in enumerate(sorted_admissions):

            codes_so_far = []
            ages_so_far = []
            time_gaps_so_far = []
            drugs_hist_so_far = []

            for j in range (i):
              vid = getHADID(sorted_admissions[j])
              drugs_hist_so_far.append(visit_medications.get(vid, []))
            current_vid = getHADID(admission)
            current_visit_drugs = visit_medications.get(current_vid, [])

            for j in range(i + 1):
              if j == 0:
                time_gaps_so_far.append(0)
              else:
                prev_admit_time = sorted_admissions[j - 1].timestamp
                curr_admit_time = sorted_admissions[j].timestamp
                time_gaps_so_far.append(bucket_time_gap((curr_admit_time - prev_admit_time).days))
              vid = getHADID(sorted_admissions[j])
              codes_so_far.append(visit_codes.get(vid, []))
              age = getAge(dob_dt, sorted_admissions[j].timestamp)
              ages_so_far.append(bucket_age(age))

            if i < len(sorted_admissions) - 1:
              discharge = convertToDT(getDischTime(admission))
              next_admit = sorted_admissions[i + 1].timestamp
              gap = (next_admit - discharge).days
              readmission = 1 if gap <= 30 else 0
            else:
                readmission = 0


            sample = {
              "patient_id": patient.patient_id,
              "codes": codes_so_far,
              "ages": ages_so_far,
              "time_gap": time_gaps_so_far,
            }
            if self.task_type == "readmission":
              sample["readmission"] = readmission

            elif self.task_type == "drug":
              sample["drugs_hist"] = drugs_hist_so_far
              sample["drugs"] = current_visit_drugs

            samples.append(sample)

        return samples

In [ ]:
task = BEHRTTask()
sample_dataset = dataset.set_task(task)